# Best Model — Bahrain Rent Prediction

**Goal:** predict monthly `rent` (BHD) for each listing in `test.csv`. **Metric:** MAE (lower is better).

**Model:** gradient boosting (`HistGradientBoostingRegressor`) trained on log(rent), with preprocessing in a `ColumnTransformer` inside a `Pipeline`.

| | MAE (BHD) |
|---|---|
| Predict the median for everything | 262.6 |
| Baseline decision tree (bedrooms + area) | 165.6 |
| **This model** (5-fold CV) | **109.5** |
| **This model** (Kaggle public leaderboard, late submission) | **102.27** |

This is the clean version. The full EDA, every decision and all the models compared are in `main.ipynb`.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error

raw_train_df = pd.read_csv('../Data/property_data.csv')

print('Train shape:', raw_train_df.shape)

Train shape: (32090, 41)


## 2. Cleaning functions
Every rule comes from the EDA in `main.ipynb`:

| Column | Rule |
|---|---|
| bedrooms | `studio` → 0, `"3+ Maid"` → 3 + `has_maid_room = 1`, `"7+"` → 7 |
| bathrooms | `"7+"` → 7, `"none"` → missing |
| area_sqm | `"1,399 sqft / 130 sqm"` → 130; under 15 sqm (1-sqm placeholders) → missing |
| governorate | `"Capital Governorate, Manama"` → `"Capital Governorate"` |
| area | filled from `city`; `"The Lagoon, Amwaj Islands"` → `"Amwaj Islands"` |
| include_w_e | inclusive / exclusive / not_selected / unknown |
| furnished, is_super_agent | lowercase / yes-no, missing → unknown |
| property_type | fewer than 50 listings in train → `Others` |
| amenities | → `amenities_count` (a number, or the length of the list) |
| title + description | → 7 keyword flags (sea view, private pool, beach, luxury, brand new, freehold, staff accommodation) |

In [2]:
def convert_bedrooms_to_number(bedrooms_value):
    if pd.isna(bedrooms_value):
        return np.nan
    bedrooms_text = str(bedrooms_value).lower()
    if bedrooms_text.startswith('studio'):
        return 0
    number_part = bedrooms_text.split('+')[0]
    return int(number_part)


def convert_bathrooms_to_number(bathrooms_value):
    if pd.isna(bathrooms_value):
        return np.nan
    bathrooms_text = str(bathrooms_value).lower()
    if bathrooms_text == 'none':
        return np.nan
    number_part = bathrooms_text.split('+')[0]
    return int(number_part)


def convert_area_to_sqm(area_value):
    if pd.isna(area_value):
        return np.nan
    area_text = str(area_value)
    sqm_part = area_text.split('/')[-1]
    sqm_part = sqm_part.replace('sqm', '').replace(',', '').strip()
    return float(sqm_part)


w_e_mapping = {'inclusive': 'inclusive', 'exclusive': 'exclusive', 'notselected': 'not_selected'}
super_agent_mapping = {True: 'yes', False: 'no'}

keyword_patterns = {
    'mentions_sea_view': 'sea view',
    'mentions_private_pool': 'private pool',
    'mentions_beach': 'beach',
    'mentions_luxury': 'luxur',
    'mentions_brand_new': 'brand new',
    'mentions_freehold': 'freehold',
    'mentions_staff_accommodation': 'accommodation|labour|labor',
}


def clean_listings(raw_df, common_property_types):
    cleaned_df = raw_df.copy()

    # rooms & size
    cleaned_df['has_maid_room'] = cleaned_df['bedrooms'].str.contains('Maid', na=False).astype(int)
    cleaned_df['bedrooms'] = cleaned_df['bedrooms'].apply(convert_bedrooms_to_number)
    cleaned_df['bathrooms'] = cleaned_df['bathrooms'].apply(convert_bathrooms_to_number)
    cleaned_df['area_sqm'] = cleaned_df['area_sqm'].apply(convert_area_to_sqm)
    cleaned_df.loc[cleaned_df['area_sqm'] < 15, 'area_sqm'] = np.nan

    # location
    cleaned_df['governorate'] = cleaned_df['governorate'].str.split(',').str[0]
    cleaned_df['area'] = cleaned_df['area'].fillna(cleaned_df['city'])
    area_has_comma = cleaned_df['area'].str.contains(',', na=False)
    cleaned_df.loc[area_has_comma, 'area'] = cleaned_df.loc[area_has_comma, 'area'].str.split(',').str[-1].str.strip()

    # categories
    cleaned_df['include_w_e'] = cleaned_df['include_w_e'].str.lower().map(w_e_mapping).fillna('unknown')
    cleaned_df['furnished'] = cleaned_df['furnished'].str.lower().fillna('unknown')
    cleaned_df['is_super_agent'] = cleaned_df['is_super_agent'].map(super_agent_mapping).fillna('unknown')
    is_common_type = cleaned_df['property_type'].isin(common_property_types)
    cleaned_df['property_type'] = cleaned_df['property_type'].where(is_common_type, 'Others')

    # amenities
    amenities_as_number = pd.to_numeric(cleaned_df['amenities'], errors='coerce')
    amenities_list_length = cleaned_df['amenities'].str.count(',') + 1
    cleaned_df['amenities_count'] = amenities_as_number.fillna(amenities_list_length)

    # keywords
    text = (cleaned_df['title'].fillna('') + ' ' + cleaned_df['description'].fillna('')).str.lower()
    for feature_name, pattern in keyword_patterns.items():
        cleaned_df[feature_name] = text.str.contains(pattern, regex=True).astype(int)

    return cleaned_df

## 3. Clean the training data
Row deletions are done on **train only** (test rows are never deleted):
1. listings with missing bedrooms or bathrooms (15 rows),
2. listings scraped twice (same URL), keeping the "Residential for Rent" copy,
3. rents of 30,000 BHD or more, which are sale prices, not monthly rents.

In [3]:
# 1. missing bedrooms / bathrooms
bedrooms_as_number = raw_train_df['bedrooms'].apply(convert_bedrooms_to_number)
bathrooms_as_number = raw_train_df['bathrooms'].apply(convert_bathrooms_to_number)
has_rooms = bedrooms_as_number.notna() & bathrooms_as_number.notna()

# property types with at least 50 listings keep their own name, the rest become 'Others'
property_type_counts = raw_train_df.loc[has_rooms, 'property_type'].value_counts()
common_property_types = property_type_counts[property_type_counts >= 50].index.tolist()
print('Property types kept:', common_property_types)

train_df = clean_listings(raw_train_df[has_rooms], common_property_types)

# 2. duplicates (same URL)
offer_priority = {'Residential for Rent': 1, 'Rent': 2, 'for rent': 3}
train_df['offer_priority'] = train_df['offer'].map(offer_priority)
train_df = train_df.sort_values('offer_priority')
is_later_copy = train_df['URL'].notna() & train_df.duplicated('URL', keep='first')
train_df = train_df[~is_later_copy].drop(columns=['offer_priority']).sort_index()

# 3. sale prices
train_df = train_df[train_df['rent'] < 30000]

print('Train shape after cleaning:', train_df.shape)

Property types kept: ['Apartment', 'Villa', 'Compound', 'Duplex', 'Penthouse', 'Whole Building', 'Townhouse']


Train shape after cleaning: (31487, 50)


## 4. Features and target

In [4]:
numeric_features = ['bedrooms', 'bathrooms', 'area_sqm', 'amenities_count', 'has_maid_room',
                    'latitude', 'longitude'] + list(keyword_patterns)
category_features = ['property_type', 'governorate', 'area', 'include_w_e', 'furnished', 'is_super_agent', 'offer']
feature_columns = numeric_features + category_features

X = train_df[feature_columns]
y = train_df['rent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((25189, 21), (6298, 21))

## 5. The pipeline
- **numeric** → median imputation (+ a "was missing" indicator) → scaling
- **categorical** → one-hot encoding; categories with fewer than 20 rows are grouped, unseen categories are handled safely
- **model** → gradient boosting trained on log(1 + rent); predictions are converted back to BHD

In [5]:
numeric_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])

category_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20, sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessing, numeric_features),
    ('category', category_preprocessing, category_features),
])

model = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1,
)

best_pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('category', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=

## 6. Evaluate

In [6]:
five_folds = KFold(n_splits=5, shuffle=True, random_state=42)

fold_mae = -cross_val_score(best_pipeline, X_train, y_train, cv=five_folds,
                            scoring='neg_mean_absolute_error', n_jobs=-1)

print('MAE per fold:', fold_mae.round(1))
print('5-fold CV MAE:', round(fold_mae.mean(), 1))

MAE per fold: [108.4 113.5 106.6 111.4 107.8]
5-fold CV MAE: 109.5


In [7]:
best_pipeline.fit(X_train, y_train)

pred_train = best_pipeline.predict(X_train)
pred_test = best_pipeline.predict(X_test)

print('Train MAE:', round(mean_absolute_error(y_train, pred_train), 1))
print('Test MAE: ', round(mean_absolute_error(y_test, pred_test), 1))

Train MAE: 97.2
Test MAE:  104.9


Train and test MAE are close, so the model generalises well to listings it hasn't seen.

## 7. Predict `test.csv` and save the submission
The final model is retrained on **all** the training data. `test.csv` goes through the same `clean_listings()`, and every test row gets a prediction.

In [8]:
best_pipeline.fit(X, y)

test_df = pd.read_csv('../Data/test.csv')
test_cleaned = clean_listings(test_df, common_property_types)

pred_submission = best_pipeline.predict(test_cleaned[feature_columns])

submission_df = pd.DataFrame({
    'Property_id': test_df['Property_id'],
    'rent': pred_submission.round(1),
})

submission_df.to_csv('../Submissions/best_model_submission.csv', index=False)

print('Submission shape:', submission_df.shape)
print('Missing predictions:', submission_df['rent'].isna().sum())
submission_df.head()

Submission shape: (3566, 2)
Missing predictions: 0


,Property_id,rent
0,25c91c33-a523-42f2-8c10-9f340005471d,478.7
1,6bcb7804-9cd7-4792-8cbe-abb4de65058a,214.2
2,7b733e67-ec21-4427-95f8-4be3d9d18f6b,1056.6
3,feefd51f-a7d6-45b0-a9eb-200d9f9821b4,351.0
4,cca286dd-ddc6-41c2-9584-f8e17461220f,478.2
